In [94]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from imblearn.over_sampling import ADASYN,SMOTE
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import nevergrad as ng
from collections import Counter

In [95]:
df = pd.read_csv("./bank-additional/bank-additional/bank-additional-full.csv",delimiter=";")
# df.drop(["default","emp.var.rate","previous","pdays","cons.conf.idx"], axis=1, inplace=True)
df.dropna()
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41188 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             41188 non-null  int64  
 1   job             41188 non-null  object 
 2   marital         41188 non-null  object 
 3   education       41188 non-null  object 
 4   default         41188 non-null  object 
 5   housing         41188 non-null  object 
 6   loan            41188 non-null  object 
 7   contact         41188 non-null  object 
 8   month           41188 non-null  object 
 9   day_of_week     41188 non-null  object 
 10  duration        41188 non-null  int64  
 11  campaign        41188 non-null  int64  
 12  pdays           41188 non-null  int64  
 13  previous        41188 non-null  int64  
 14  poutcome        41188 non-null  object 
 15  emp.var.rate    41188 non-null  float64
 16  cons.price.idx  41188 non-null  float64
 17  cons.conf.idx   41188 non-null 

# Preprocessing

In [96]:
# Feature engineering and dropping columns
df['duration'] = (df['duration']/60).round()

# df.drop(['day_of_week','default','pdays'], axis=1, inplace=True)

df['y'] = df['y'].map({'yes': 1, 'no': 0})

In [97]:


# OUTLIER HANDLING duration (IMPUTATION)
Q1_duration = df['duration'].quantile(0.25)
Q3_duration = df['duration'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['duration'].median()
df.loc[df['duration'] < lower_bound_duration, 'duration'] = median_duration
df.loc[df['duration'] > upper_bound_duration, 'duration'] = median_duration

# OUTLIER HANDLING campaign (IMPUTATION)
Q1_duration = df['campaign'].quantile(0.25)
Q3_duration = df['campaign'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['campaign'].median()
df.loc[df['campaign'] < lower_bound_duration, 'campaign'] = median_duration
df.loc[df['campaign'] > upper_bound_duration, 'campaign'] = median_duration

# OUTLIER HANDLING age (IMPUTATION)
Q1_duration = df['age'].quantile(0.25)
Q3_duration = df['age'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['age'].median()
df.loc[df['age'] < lower_bound_duration, 'age'] = median_duration
df.loc[df['age'] > upper_bound_duration, 'age'] = median_duration



# OUTLIER HANDLING pdays (IMPUTATION)
Q1_duration = df['pdays'].quantile(0.25)
Q3_duration = df['pdays'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['pdays'].median()
df.loc[df['pdays'] < lower_bound_duration, 'pdays'] = median_duration
df.loc[df['pdays'] > upper_bound_duration, 'pdays'] = median_duration



# OUTLIER HANDLING previous (IMPUTATION)
Q1_duration = df['previous'].quantile(0.25)
Q3_duration = df['previous'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['previous'].median()
df.loc[df['previous'] < lower_bound_duration, 'previous'] = median_duration
df.loc[df['previous'] > upper_bound_duration, 'previous'] = median_duration



# OUTLIER HANDLING emp.var.rate (IMPUTATION)
Q1_duration = df['emp.var.rate'].quantile(0.25)
Q3_duration = df['emp.var.rate'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['emp.var.rate'].median()
df.loc[df['emp.var.rate'] < lower_bound_duration, 'emp.var.rate'] = median_duration
df.loc[df['emp.var.rate'] > upper_bound_duration, 'emp.var.rate'] = median_duration


# OUTLIER HANDLING cons.price.idx (IMPUTATION)
Q1_duration = df['cons.price.idx'].quantile(0.25)
Q3_duration = df['cons.price.idx'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['cons.price.idx'].median()
df.loc[df['cons.price.idx'] < lower_bound_duration, 'cons.price.idx'] = median_duration
df.loc[df['cons.price.idx'] > upper_bound_duration, 'cons.price.idx'] = median_duration





# OUTLIER HANDLING cons.conf.idx (IMPUTATION)
Q1_duration = df['cons.conf.idx'].quantile(0.25)
Q3_duration = df['cons.conf.idx'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['cons.conf.idx'].median()
df.loc[df['cons.conf.idx'] < lower_bound_duration, 'cons.conf.idx'] = median_duration
df.loc[df['cons.conf.idx'] > upper_bound_duration, 'cons.conf.idx'] = median_duration


# OUTLIER HANDLING euribor3m (IMPUTATION)
Q1_duration = df['euribor3m'].quantile(0.25)
Q3_duration = df['euribor3m'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['euribor3m'].median()
df.loc[df['euribor3m'] < lower_bound_duration, 'euribor3m'] = median_duration
df.loc[df['euribor3m'] > upper_bound_duration, 'euribor3m'] = median_duration

# OUTLIER HANDLING nr.employed (IMPUTATION)
Q1_duration = df['nr.employed'].quantile(0.25)
Q3_duration = df['nr.employed'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = max(0, Q1_duration - 1.5 * IQR_duration)
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
median_duration = df['nr.employed'].median()
df.loc[df['nr.employed'] < lower_bound_duration, 'nr.employed'] = median_duration
df.loc[df['nr.employed'] > upper_bound_duration, 'nr.employed'] = median_duration

df = df[df['job'] != 'unknown']
df = df[df['marital'] != 'unknown']
df = df[df['education'] != 'unknown']
df = df[df['housing'] != 'unknown']
df = df[df['loan'] != 'unknown']
df = df[df['contact'] != 'unknown']
df = df[df['month'] != 'unknown']
df = df[df['day_of_week'] != 'unknown']
df = df[df['poutcome'] != 'unknown']


'''
DELETION HANDLING
# OUTLIER HANDLING FOR DURATION
Q1_duration = df['duration'].quantile(0.25)
Q3_duration = df['duration'].quantile(0.75)
IQR_duration = Q3_duration - Q1_duration
lower_bound_duration = Q1_duration - 1.5 * IQR_duration
upper_bound_duration = Q3_duration + 1.5 * IQR_duration
df = df[(df['duration'] >= max(0, lower_bound_duration)) & 
        (df['duration'] <= upper_bound_duration)]

# OUTLIER HANDLING FOR CAMPAIGN
Q1_campaign = df['campaign'].quantile(0.25)
Q3_campaign = df['campaign'].quantile(0.75)
IQR_campaign = Q3_campaign - Q1_campaign
lower_bound_campaign = Q1_campaign - 1.5 * IQR_campaign
upper_bound_campaign = Q3_campaign + 1.5 * IQR_campaign
df = df[(df['campaign'] >= max(1, lower_bound_campaign)) & 
        (df['campaign'] <= upper_bound_campaign)]

# OUTLIER HANDLING FOR AGE
Q1_age = df['age'].quantile(0.25)
Q3_age = df['age'].quantile(0.75)
IQR_age = Q3_age - Q1_age
lower_bound_age = Q1_age - 1.5 * IQR_age
upper_bound_age = Q3_age + 1.5 * IQR_age
df = df[(df['age'] >= max(1, lower_bound_campaign)) & 
        (df['age'] <= upper_bound_campaign)]

# OUTLIER HANDLING FOR pdays
Q1_pdays = df['pdays'].quantile(0.25)
Q3_pdays = df['pdays'].quantile(0.75)
IQR_pdays = Q3_pdays - Q1_pdays
lower_bound_pdays = Q1_pdays - 1.5 * IQR_pdays
upper_bound_pdays = Q3_pdays + 1.5 * IQR_pdays
df = df[(df['pdays'] >= max(1, lower_bound_pdays)) & 
        (df['pdays'] <= upper_bound_pdays)]

# OUTLIER HANDLING FOR previous
Q1_previous = df['previous'].quantile(0.25)
Q3_previous = df['previous'].quantile(0.75)
IQR_previous = Q3_previous - Q1_previous
lower_bound_previous = Q1_previous - 1.5 * IQR_previous
upper_bound_previous = Q3_previous + 1.5 * IQR_previous
df = df[(df['previous'] >= max(1, lower_bound_previous)) & 
        (df['previous'] <= upper_bound_previous)]

# OUTLIER HANDLING FOR emp.var.rate
Q1_emp = df['emp.var.rate'].quantile(0.25)
Q3_emp = df['emp.var.rate'].quantile(0.75)
IQR_emp = Q3_emp - Q1_emp
lower_bound_emp = Q1_emp - 1.5 * IQR_emp
upper_bound_emp = Q3_emp + 1.5 * IQR_emp
df = df[(df['emp.var.rate'] >= max(1, lower_bound_emp)) & 
        (df['emp.var.rate'] <= upper_bound_emp)]

# OUTLIER HANDLING FOR cons.price.idx
Q1_cons = df['cons.price.idx'].quantile(0.25)
Q3_cons = df['cons.price.idx'].quantile(0.75)
IQR_cons = Q3_cons - Q1_cons
lower_bound_cons = Q1_cons - 1.5 * IQR_cons
upper_bound_cons = Q3_cons + 1.5 * IQR_cons
df = df[(df['cons.price.idx'] >= max(1, lower_bound_cons)) & 
        (df['cons.price.idx'] <= upper_bound_cons)]

# OUTLIER HANDLING FOR cons.conf.idx
Q1_conf = df['cons.conf.idx'].quantile(0.25)
Q3_conf = df['cons.conf.idx'].quantile(0.75)
IQR_conf = Q3_conf - Q1_conf
lower_bound_conf = Q1_conf - 1.5 * IQR_conf
upper_bound_conf = Q3_conf + 1.5 * IQR_conf
df = df[(df['cons.conf.idx'] >= max(1, lower_bound_conf)) & 
        (df['cons.conf.idx'] <= upper_bound_conf)]

# OUTLIER HANDLING FOR euribor3m
Q1_euribor3m = df['euribor3m'].quantile(0.25)
Q3_euribor3m = df['euribor3m'].quantile(0.75)
IQR_euribor3m = Q3_euribor3m - Q1_euribor3m
lower_bound_euribor3m = Q1_euribor3m - 1.5 * IQR_euribor3m
upper_bound_euribor3m = Q3_euribor3m + 1.5 * IQR_euribor3m
df = df[(df['euribor3m'] >= max(1, lower_bound_euribor3m)) & 
        (df['euribor3m'] <= upper_bound_euribor3m)]

# OUTLIER HANDLING FOR nr.employed
Q1_employed = df['nr.employed'].quantile(0.25)
Q3_employed = df['nr.employed'].quantile(0.75)
IQR_employed = Q3_employed - Q1_employed
lower_bound_employed = Q1_employed - 1.5 * IQR_employed
upper_bound_employed = Q3_employed + 1.5 * IQR_employed
df = df[(df['nr.employed'] >= max(1, lower_bound_employed)) & 
        (df['nr.employed'] <= upper_bound_employed)]   

# unknown category
df = df[df['education'] != 'unknown']
df = df[df['job'] != 'unknown']

'''

"\nDELETION HANDLING\n# OUTLIER HANDLING FOR DURATION\nQ1_duration = df['duration'].quantile(0.25)\nQ3_duration = df['duration'].quantile(0.75)\nIQR_duration = Q3_duration - Q1_duration\nlower_bound_duration = Q1_duration - 1.5 * IQR_duration\nupper_bound_duration = Q3_duration + 1.5 * IQR_duration\ndf = df[(df['duration'] >= max(0, lower_bound_duration)) & \n        (df['duration'] <= upper_bound_duration)]\n\n# OUTLIER HANDLING FOR CAMPAIGN\nQ1_campaign = df['campaign'].quantile(0.25)\nQ3_campaign = df['campaign'].quantile(0.75)\nIQR_campaign = Q3_campaign - Q1_campaign\nlower_bound_campaign = Q1_campaign - 1.5 * IQR_campaign\nupper_bound_campaign = Q3_campaign + 1.5 * IQR_campaign\ndf = df[(df['campaign'] >= max(1, lower_bound_campaign)) & \n        (df['campaign'] <= upper_bound_campaign)]\n\n# OUTLIER HANDLING FOR AGE\nQ1_age = df['age'].quantile(0.25)\nQ3_age = df['age'].quantile(0.75)\nIQR_age = Q3_age - Q1_age\nlower_bound_age = Q1_age - 1.5 * IQR_age\nupper_bound_age = Q3_age 

In [98]:
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0


In [99]:
# X & y
X = df.drop('y', axis=1)
y = df['y']

In [100]:
y.head()

0    0
1    0
2    0
3    0
4    0
Name: y, dtype: int64

In [101]:
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
Index: 38245 entries, 0 to 41187
Data columns (total 21 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   age             38245 non-null  int64  
 1   job             38245 non-null  object 
 2   marital         38245 non-null  object 
 3   education       38245 non-null  object 
 4   default         38245 non-null  object 
 5   housing         38245 non-null  object 
 6   loan            38245 non-null  object 
 7   contact         38245 non-null  object 
 8   month           38245 non-null  object 
 9   day_of_week     38245 non-null  object 
 10  duration        38245 non-null  float64
 11  campaign        38245 non-null  int64  
 12  pdays           38245 non-null  int64  
 13  previous        38245 non-null  int64  
 14  poutcome        38245 non-null  object 
 15  emp.var.rate    38245 non-null  float64
 16  cons.price.idx  38245 non-null  float64
 17  cons.conf.idx   38245 non-null  floa

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,1,999,0,nonexistent,1.1,93.994,-41.8,4.857,5191.0,0


In [102]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print(f"Original X_train shape: {X_train.shape}")


Original X_train shape: (26771, 20)


In [103]:
encoder_cols = ['education', 'job', 'month', 'poutcome','marital','housing','loan','contact','day_of_week']

# encoder_cols = ['month','loan','day_of_week']

target_encoder = TargetEncoder(cols=encoder_cols, smoothing=1.0) 

X_train_encoded = target_encoder.fit_transform(X_train, y_train)
X_test_encoded = target_encoder.transform(X_test)

print(f"Encoded X_train shape: {X_train_encoded.shape}")
X_trainNB, X_testNB, y_trainNB, y_testNB = X_train_encoded.copy(), X_test_encoded.copy(), y_train.copy(), y_test.copy()


Encoded X_train shape: (26771, 20)


In [105]:
y_train.head()

23133    0
27112    0
8866     1
8878     0
6402     0
Name: y, dtype: int64

In [104]:
ismote = SMOTE(random_state=42, sampling_strategy = "auto", k_neighbors = 10)
X_train_res, y_train_res = ismote.fit_resample(X_train_encoded, y_train)

X_train_res_nb, y_train_res_nb = X_train_res, y_train_res # Store for Naive Bayes final fitting

X_train_res_nb_bot, y_train_res_nb_bot = X_train_res_nb.copy(), y_train_res_nb.copy()

print(f"\nResampled X_train shape: {X_train_res.shape}")
print(f"Class distribution after ADASYN: {Counter(y_train_res)}")
print(X_train_res_nb_bot.info())
print(y_train_res_nb_bot.info())

ValueError: could not convert string to float: 'no'

In [ ]:
X_test.head()

,age,job,marital,education,housing,loan,contact,month,day_of_week,duration,campaign,poutcome,cons.price.idx,euribor3m,nr.employed
37898,38,admin.,single,university.degree,yes,no,telephone,sep,tue,8.0,1,nonexistent,92.379,0.819,5017.5
6299,55,technician,married,basic.6y,yes,no,telephone,may,tue,8.0,2,nonexistent,93.994,4.857,5191.0
13,57,housemaid,divorced,basic.4y,yes,no,telephone,may,mon,5.0,1,nonexistent,93.994,4.857,5191.0
9682,29,self-employed,married,high.school,no,no,telephone,jun,mon,4.0,6,nonexistent,94.465,4.961,5228.1
18954,44,admin.,divorced,high.school,yes,no,cellular,aug,mon,2.0,2,nonexistent,93.444,4.970,5228.1


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

clf = DecisionTreeClassifier(random_state=42)   # reproducible tree structure
clf.fit(X_train_res, y_train_res)

y_pred = clf.predict(X_test_encoded)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred))
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
 
feature_importances = pd.Series(clf.feature_importances_, index=X.columns)
feature_importances = feature_importances.sort_values(ascending=False)

print("\nFeature Importances:")
print(feature_importances)


Accuracy: 0.8557

Classification Report:
              precision    recall  f1-score   support

           0       0.93      0.91      0.92     10197
           1       0.37      0.44      0.40      1277

    accuracy                           0.86     11474
   macro avg       0.65      0.67      0.66     11474
weighted avg       0.87      0.86      0.86     11474


Confusion Matrix:
[[9256  941]
 [ 715  562]]

Feature Importances:
duration          0.366845
nr.employed       0.119083
day_of_week       0.093756
month             0.062713
age               0.050656
housing           0.046676
cons.price.idx    0.041798
contact           0.039416
euribor3m         0.038658
education         0.035329
job               0.029032
campaign          0.019947
poutcome          0.019884
loan              0.019546
marital           0.016661
dtype: float64
